In [1]:
from playwright.async_api import async_playwright
import time
import os
async def login():
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        await page.goto("https://www.naukri.com/")
        print("Login manually in this window...")

        input("Press ENTER after login is complete...")

        await context.storage_state(path="naukri_state.json")
        
apply = await login()

Login manually in this window...


In [ ]:
async def scrape_jobs():
    jobs_data = []
    category = "data-science-internship"  # Change this to your desired category

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="state.json")
        
        page = await context.new_page()
        # search_url = f"https://internshala.com/internships/{category}"
        # await page.goto(
        #     "https://internshala.com/internships/"+category, 
        #     wait_until="networkidle"
        # )

In [2]:
import asyncio
from urllib.parse import quote
from playwright.async_api import async_playwright


from urllib.parse import quote


def construct_naukri_url(job_titles: list[str], experience: int, role_type: str) -> str:
    """
    Build Naukri search URL correctly for job or internship.
    """

    role_type = role_type.lower().strip()

    # Clean titles
    titles = [t.strip().lower() for t in job_titles]

    # -------- PATH PART --------
    # hyphen separated
    slug_part = "-".join(t.replace(" ", "-") for t in titles)

    if role_type == "internship":
        path = f"{slug_part}-internship-jobs"
    else:
        path = f"{slug_part}-jobs"

    # -------- QUERY PART --------
    # comma separated
    keyword_string = ", ".join(titles)

    if role_type == "internship":
        keyword_string += " internship"

    encoded_keywords = quote(keyword_string)

    # -------- FINAL URL --------
    base = f"https://www.naukri.com/{path}"

    if role_type == "internship":
        url = (
            f"{base}"
            f"?k={encoded_keywords}"
            f"&experience={experience}"
            f"&qproductJobSource=2"
            f"&qinternshipFlag=true"
            f"&naukriCampus=true"
        )
    else:
        url = (
            f"{base}"
            f"?k={encoded_keywords}"
            f"&experience={experience}"
            f"&qproductJobSource=2"
            f"&naukriCampus=true"
        )

    return url

import asyncio
from playwright.async_api import async_playwright


async def scrape_naukri_jobs(page):
    jobs_data = []

    # Wait until job cards load
    await page.wait_for_selector(".srp-jobtuple-wrapper", timeout=15000)

    cards = await page.query_selector_all(".srp-jobtuple-wrapper")

    for card in cards:
        try:
            # Title
            title_el = await card.query_selector("h2 a.title")
            title = await title_el.inner_text() if title_el else None
            link = await title_el.get_attribute("href") if title_el else None

            # Company
            company_el = await card.query_selector(".comp-name")
            company = await company_el.inner_text() if company_el else None

            # Experience (NOT duration for jobs)
            exp_el = await card.query_selector(".exp-wrap span span")
            experience = await exp_el.inner_text() if exp_el else None

            # Salary / Stipend
            sal_el = await card.query_selector(".sal-wrap span span")
            stipend = await sal_el.inner_text() if sal_el else None

            # Location
            loc_el = await card.query_selector(".loc-wrap span span")
            location = await loc_el.inner_text() if loc_el else None

            # Posted / Starts in
            posted_el = await card.query_selector(".job-post-day")
            starts_in = await posted_el.inner_text() if posted_el else None

            # Description (NOT present in listing card)
            # Naukri listing page doesn't contain full description in card.
            description = None

            jobs_data.append({
                "title": title,
                "link": link,
                "company": company,
                "location": location,
                "salary_or_stipend": stipend,
                "experience_or_duration": experience,
                "description": description,
                "starts_in": starts_in
            })

        except Exception as e:
            print("Error parsing card:", e)

    return jobs_data

async def run_naukri_search(job_titles, experience, role_type):
    constructed_urls = []

    # Build single combined URL
    url = construct_naukri_url(job_titles, experience, role_type)
    constructed_urls.append(url)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="naukri_state.json")
        page = await context.new_page()

        # Navigate to constructed search URL
        await page.goto(url, wait_until="domcontentloaded")
        print("Navigated to:", url)
        global jobs
        jobs = await scrape_naukri_jobs(page)
        print(f"Fetched {len(jobs)} jobs")
        for job in jobs:
            print(job)

        await browser.close()
        
        await asyncio.sleep(5)  # Let page load visually
        await browser.close()

    return constructed_urls

# Preffered job titles. Primary job role and other preffered job roles that are typed during the onboarding will be added here.
job_titles = ["ai ml engineer", "gen ai developer", "data scientist"]
experience = 0
role_type = "job"  # or "job"

await run_naukri_search(job_titles, experience, role_type)

Navigated to: https://www.naukri.com/ai-ml-engineer-gen-ai-developer-data-scientist-jobs?k=ai%20ml%20engineer%2C%20gen%20ai%20developer%2C%20data%20scientist&experience=0&qproductJobSource=2&naukriCampus=true
Fetched 20 jobs
{'title': 'ML / Data Scientist / Gen AI Specialist', 'link': 'https://www.naukri.com/job-listings-ml-data-scientist-gen-ai-specialist-gibots-pune-0-to-4-years-030724500606', 'company': 'Gibots', 'location': 'Pune', 'salary_or_stipend': None, 'experience_or_duration': '0-4 Yrs', 'description': None, 'starts_in': '3+ weeks ago'}
{'title': 'Gen AI Engineer', 'link': 'https://www.naukri.com/job-listings-gen-ai-engineer-exotel-bengaluru-0-to-2-years-270126015270', 'company': 'Exotel', 'location': 'Bengaluru', 'salary_or_stipend': '5-15 Lacs PA', 'experience_or_duration': '0-2 Yrs', 'description': None, 'starts_in': '3+ weeks ago'}
{'title': 'Data Scientist', 'link': 'https://www.naukri.com/job-listings-data-scientist-persolkelly-india-gurugram-0-to-1-years-200126014641'

['https://www.naukri.com/ai-ml-engineer-gen-ai-developer-data-scientist-jobs?k=ai%20ml%20engineer%2C%20gen%20ai%20developer%2C%20data%20scientist&experience=0&qproductJobSource=2&naukriCampus=true']

In [4]:
jobs

[{'title': 'Junior AI/ML Engineer ( Fresher)',
  'link': 'https://www.naukri.com/job-listings-junior-ai-ml-engineer-fresher-reizend-pvt-ltd-thiruvananthapuram-0-to-1-years-110226038080',
  'company': 'Reizend It Consultants',
  'location': 'Thiruvananthapuram',
  'salary_or_stipend': '4-6 Lacs PA',
  'experience_or_duration': '0-1 Yrs',
  'description': None,
  'starts_in': '2 weeks ago'},
 {'title': 'ML / Data Scientist / Gen AI Specialist',
  'link': 'https://www.naukri.com/job-listings-ml-data-scientist-gen-ai-specialist-gibots-pune-0-to-4-years-030724500606',
  'company': 'Gibots',
  'location': 'Pune',
  'salary_or_stipend': None,
  'experience_or_duration': '0-4 Yrs',
  'description': None,
  'starts_in': '3+ weeks ago'},
 {'title': 'Gen AI Engineer',
  'link': 'https://www.naukri.com/job-listings-gen-ai-engineer-exotel-bengaluru-0-to-2-years-270126015270',
  'company': 'Exotel',
  'location': 'Bengaluru',
  'salary_or_stipend': '5-15 Lacs PA',
  'experience_or_duration': '0-2 Yr

In [3]:
jobs[1]

{'title': 'Gen AI Engineer',
 'link': 'https://www.naukri.com/job-listings-gen-ai-engineer-exotel-bengaluru-0-to-2-years-270126015270',
 'company': 'Exotel',
 'location': 'Bengaluru',
 'salary_or_stipend': '5-15 Lacs PA',
 'experience_or_duration': '0-2 Yrs',
 'description': None,
 'starts_in': '3+ weeks ago'}

# Applying to a page

In [ ]:
jobs[3]['link']

'https://www.naukri.com/job-listings-ai-ml-engineer-eidiko-systems-integrators-hyderabad-0-to-3-years-260226017010'

In [8]:
import asyncio
from playwright.async_api import async_playwright


async def handle_case_1(page):
    """Direct Apply Case"""
    print("CASE 1: Direct Apply detected")

    await page.click("#apply-button")

    # Wait for navigation / success message
    await page.wait_for_load_state("networkidle")

    try:
        await page.wait_for_selector("span.apply-message", timeout=5000)
        message = await page.locator("span.apply-message").inner_text()

        if "successfully applied" in message.lower():
            print("Successfully applied ✅")
            return True

    except:
        print("Apply message not found.")

    return False


async def extract_latest_question(page):
    """Extract last bot question from chat"""
    messages = page.locator("li.botItem .botMsg span")
    count = await messages.count()

    if count == 0:
        return None

    last_question = await messages.nth(count - 1).inner_text()
    return last_question.strip()


async def handle_radio_question(page):
    """Handle radio button type question"""

    print("Radio options detected")

    radio_inputs = page.locator("input[type='radio']")
    count = await radio_inputs.count()

    options = []
    for i in range(count):
        value = await radio_inputs.nth(i).get_attribute("value")
        options.append(value)

    print("Options:")
    for idx, opt in enumerate(options):
        print(f"{idx + 1}. {opt}")

    choice = input("Enter option number: ").strip()

    try:
        selected_index = int(choice) - 1
        selected_value = options[selected_index]
    except:
        print("Invalid input, skipping.")
        return

    await page.locator(f"input[type='radio'][value='{selected_value}']").click()

    await page.click("div.sendMsg")
    await asyncio.sleep(2)


async def handle_text_question(page):
    """Handle text input question"""

    question = await extract_latest_question(page)

    if not question:
        return

    print("\nQuestion:", question)

    answer = input("Your Answer: ").strip()

    input_box = page.locator("div.textArea[contenteditable='true']")
    await input_box.click()
    await input_box.fill(answer)

    await page.click("div.sendMsg")

    # Wait 2 seconds before fetching next question
    await asyncio.sleep(2)


async def handle_case_2(page):
    """Chat based application flow"""
    print("CASE 2: Chat based application detected")

    await page.click("#walkin-button")

    await asyncio.sleep(3)  # Wait for chat to load

    while True:
        # Check for final success message
        apply_message = page.locator("span.apply-message")
        if await apply_message.count() > 0:
            text = await apply_message.inner_text()
            print("\nFinal Message:", text)

            if "recorded successfully" in text.lower():
                print("Application Completed ✅")
                break

        # Check if radio buttons exist
        if await page.locator("input[type='radio']").count() > 0:
            await handle_radio_question(page)
        else:
            await handle_text_question(page)


async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="naukri_state.json")
        page = await context.new_page()

        await page.goto(jobs[3]['link'])

        await page.wait_for_load_state("networkidle")

        # Detect which case
        if await page.locator("#apply-button").count() > 0:
            success = await handle_case_1(page)
            if success:
                return

        if await page.locator("#walkin-button").count() > 0:
            await handle_case_2(page)

        print("Done.")


await main()

CASE 2: Chat based application detected


TargetClosedError: Page.click: Target page, context or browser has been closed
Call log:
  - waiting for locator("#walkin-button")
    - locator resolved to <button id="walkin-button" class="styles_walkin-button__gRooP walkin-button">I am interested</button>
  - attempting click action
    2 × waiting for element to be visible, enabled and stable
      - element is not visible
    - retrying click action
    - waiting 20ms
    2 × waiting for element to be visible, enabled and stable
      - element is not visible
    - retrying click action
      - waiting 100ms
    39 × waiting for element to be visible, enabled and stable
       - element is not visible
     - retrying click action
       - waiting 500ms
